# 01 · Why LLM inference is its own problem
Level L1 · Part 1 · core · ~25 min · builds on none

## Goal
By the end of this lesson you can explain why serving an LLM costs more per request, and fails differently, than a web API. You can name the two phases (prefill and decode) and the KV cache that drive that cost.

## Why it matters
- **Without it:** you run the model like a web handler, one request at a time, recomputing every answer from the start. A 200-token answer pushes the whole prompt through the network 200 times. One long request holds the GPU while others wait, and a burst of users can crash it with an out-of-memory error.
- **With it:** you know what a request costs in tokens and memory, so you can size a GPU and set limits.
- **When you don't need it:** if a hosted provider runs the model and you only call its API, the provider does this work. You still pay per token.

## The idea
A web API call is a vending machine: the same button, the same cost, a few milliseconds. An LLM answer is a chef who writes one **token** (a chunk of text, about three-quarters of a word) at a time, and before each token rereads the order so far. The **KV cache** is the chef's notes, so earlier text does not need reading again. The **weights** are the model's learned numbers, about 16 GB for an 8B model in FP16. The notes take GPU memory for as long as the request is open. So serving an LLM is a memory and scheduling problem, not a request-count problem.

## Picture
Without an inference engine, every request is a full recompute on one shared model. With one, the engine batches requests, stores the notes in blocks and reuses the shared policy prompt.

Previews, each taught later: the **gateway** (lesson 29), the engine's **scheduler** and batching (lesson 12), **blocks** of KV memory (lesson 11), and the **policy prefix** reused across requests (lesson 14).

```mermaid
flowchart LR
  U1[Client] --> H["Plain Python handler"]
  H --> M1["Model: one request at a time"]
  M1 -->|"recompute all tokens every step"| M1
  M1 --> R1["Whole answer after N steps"]
```

```mermaid
flowchart LR
  U2[Client] --> G["Gateway: queue and limits"]
  G --> E["Engine scheduler"]
  E --> B["Batch: many requests per step"]
  B --> K["KV cache blocks on the GPU; policy prefix stored once"]
  B --> W["Model weights, read once per step"]
  E -->|"streamed tokens"| U2
```

## How it simplifies the architecture
| | Without an inference engine | With an inference engine |
|---|---|---|
| Batching | your code, or none (one request at a time) | the engine's scheduler, every step |
| KV memory | none: the work is recomputed instead | the engine's block allocator |
| Reusing a shared prompt | none | prefix caching (lesson 14) |
| Streaming tokens | you write it | built in |
| Failure points | out of memory, one long request blocks the rest | you still own the gateway, limits and capacity |

The engine removes the glue for batching and memory. It does **not** fix an undersized GPU, a bad prompt, or a quality drop after a config change. Those need capacity maths (lesson 05, lesson 26) and evals (lesson 35).

### When you don't need an engine
A hosted API, or a model small enough to answer one request at a time, can be enough. Alternatives: a hosted provider API; llama.cpp or Ollama for local use; vLLM, SGLang or TensorRT-LLM for self-hosting. Lesson 15 compares the engines.

## Already know this?
Optional self-check for experienced readers. Four questions cover Part 1 (lessons 01 to 09). If you can't answer them, carry on: the Step by step and the Recap cover them.
1. How many bytes does one token of KV cache take for FP16 Llama 3.1 8B? (lesson 03)
2. Why is decode slower per token than prefill per token? (lesson 06)
3. What is the difference between TTFT and TPOT? (lesson 07)
4. Why does one long request raise the latency of the others? (lessons 12 and 33)

## Step by step
This is a walkthrough. Each step runs a small piece of arithmetic that uses the same numbers as the rest of the course. Nothing here needs a GPU, a key or a download.

**Step 1: set the example numbers.** The Brightside support desk sends every request with the same 1,500-token policy prompt, then a 100-token question and a 200-token answer. The question and answer lengths are assumptions. The KV size per token uses the public Llama 3.1 8B config: 32 layers, 8 KV heads, head size 128, 2 bytes per value (FP16). Those shape numbers are explained in lesson 03.

Run this notebook from its own folder (`notebooks/`). The first code line then finds `tut_env.py` in the course folder.

In [1]:
import sys; sys.path.insert(0, "..")   # the course folder, so tut_env.py can be imported
from tut_env import env  # noqa: F401  (reads .env settings; used by the last cell)

POLICY_TOKENS = 1_500    # shared policy prompt, the same for every request
QUESTION_TOKENS = 100    # the customer's question (assumed)
ANSWER_TOKENS = 200      # the answer (assumed)
LAYERS, KV_HEADS, HEAD_DIM, BYTES = 32, 8, 128, 2   # model shape (lesson 03); FP16 = 2 bytes
KV_BYTES_PER_TOKEN = 2 * LAYERS * KV_HEADS * HEAD_DIM * BYTES   # the 2 = one key and one value
print(f"KV cache per token: {KV_BYTES_PER_TOKEN:,} bytes = {KV_BYTES_PER_TOKEN // 1024} KiB")

KV cache per token: 131,072 bytes = 128 KiB


**Step 2: count the work without a KV cache.** Without notes, each new token makes the model run the whole sequence again. With notes, the prompt runs once and each new token runs once. The counts below are token passes through the network. They are counted, not timed.

In [2]:
PROMPT = POLICY_TOKENS + QUESTION_TOKENS

def token_passes(prompt, answer, kv_cache):
    """Tokens the network processes to write the whole answer."""
    if kv_cache:
        return prompt + answer                       # prefill once, then one token per step
    return sum(prompt + i for i in range(answer))    # every step reruns the whole sequence

no_cache = token_passes(PROMPT, ANSWER_TOKENS, kv_cache=False)
cached = token_passes(PROMPT, ANSWER_TOKENS, kv_cache=True)
print(f"without KV cache: {no_cache:,} token passes")
print(f"with KV cache:    {cached:,} token passes")
print(f"ratio: {no_cache / cached:,.0f}x (counted, not measured)")

without KV cache: 339,900 token passes
with KV cache:    1,800 token passes
ratio: 189x (counted, not measured)


**What you should see:** about 340,000 token passes without notes and 1,800 with them.

Where the 340,000 comes from: step 1 processes 1,600 tokens, step 2 processes 1,601, and so on up to step 200, which processes 1,799. The average is about 1,700, so 200 × 1,700 is about 340,000. With notes, the 1,600 prompt tokens run once and the 200 new tokens run once each, which is 1,800. The gap grows with answer length.

**Step 3: follow one request through its phases.** The prompt is processed in one parallel pass (prefill). Then the model writes one token per step (decode). Each decode step reads the weights and every cached token.

In [3]:
MIB = 2**20

def lifecycle(prompt, answer):
    print(f"1. prefill: {prompt:,} prompt tokens in one parallel pass")
    print(f"   KV cache now holds {prompt:,} tokens = {prompt * KV_BYTES_PER_TOKEN / MIB:,.0f} MiB")
    for step in range(1, answer + 1):
        if step in (1, 2, answer):
            print(f"2. decode step {step}: 1 new token, reads {prompt + step - 1:,} cached tokens, "
                  f"cache now {prompt + step:,}")
        elif step == 3:
            print("   ...")

lifecycle(PROMPT, ANSWER_TOKENS)

1. prefill: 1,600 prompt tokens in one parallel pass
   KV cache now holds 1,600 tokens = 200 MiB
2. decode step 1: 1 new token, reads 1,600 cached tokens, cache now 1,601
2. decode step 2: 1 new token, reads 1,601 cached tokens, cache now 1,602
   ...
2. decode step 200: 1 new token, reads 1,799 cached tokens, cache now 1,800


**What you should see:** prefill fills the cache in one pass, then each decode step adds one token. The first token the customer sees comes at the end of prefill. That time is the **time to first token (TTFT)**. The time between later tokens is the **time per output token (TPOT)**.

**Step 4: memory decides how many requests fit.** The KV cache has to fit beside the weights. The example assumes an 80 GiB GPU, uses 92% of its memory (the vLLM default), and loads the 8B model at 16 GB in FP16. Two other uses of memory are ignored: **activations** (temporary buffers during each forward pass) and **fragmentation** (small gaps that cannot be used). So the counts are an upper bound.

In [4]:
GIB = 2**30
GPU_BYTES = 80 * GIB
USABLE = 0.92                                          # share of GPU memory vLLM may use (its default)
WEIGHT_BYTES = 16e9                                    # 8B parameters x 2 bytes (FP16)

kv_budget = GPU_BYTES * USABLE - WEIGHT_BYTES          # memory left for the KV cache
kv_per_request_full = (PROMPT + ANSWER_TOKENS) * KV_BYTES_PER_TOKEN   # all 1,800 tokens of one request
shared_prefix = POLICY_TOKENS * KV_BYTES_PER_TOKEN     # the policy prompt, stored once
kv_per_request_private = (QUESTION_TOKENS + ANSWER_TOKENS) * KV_BYTES_PER_TOKEN   # question + answer = 300 tokens

print(f"KV budget after the weights: {kv_budget / GIB:.1f} GiB")
# // is floor division: it counts whole requests only
print(f"requests, each with its own copy of the prompt: {kv_budget // kv_per_request_full:,.0f}")
print(f"requests, policy prefix shared by all:          {(kv_budget - shared_prefix) // kv_per_request_private:,.0f}")

KV budget after the weights: 58.7 GiB
requests, each with its own copy of the prompt: 267
requests, policy prefix shared by all:          1,597


**What you should see:** about 270 requests if every request keeps its own copy of the policy prompt, and about 1,600 if all requests share the same 1,500-token policy prefix. This sharing is an assumption of the example: every request uses the same prefix. These are estimates from the formulas above. Lesson 05 does this sizing properly, and lesson 14 shows prefix caching.


**Step 5: decode is limited by memory, not maths.** A **memory-bound** step is limited by how fast data is read from GPU memory, not by maths. A **batch** means several requests processed in one step. At batch 1, each decode step reads all 16 GB of weights to make one token.

The step times below use spec-sheet numbers for the NVIDIA H100 SXM (the H100 in its SXM socket): 3.35 TB/s of **HBM** (high-bandwidth memory, the GPU's main memory) and 989 **dense** TFLOPS of FP16 (trillion floating-point operations per second; dense means without the sparsity doubling, which is half of the 1,979 on NVIDIA's page). Source: https://www.nvidia.com/en-us/data-center/h100/ . The results are estimates. Real engines add overhead.

In [5]:
PARAMS = 8e9                   # Llama 3.1 8B parameters
H100_BW = 3.35e12              # bytes per second of HBM
H100_FLOPS = 989e12            # dense FP16 FLOP per second
CONTEXT = 1_800                # tokens per request in the example
WEIGHT_READ = PARAMS * 2       # every decode step reads all weights once (FP16 = 2 bytes)

for batch in (1, 64):
    t_mem = (WEIGHT_READ + batch * CONTEXT * KV_BYTES_PER_TOKEN) / H100_BW   # weights + KV reads, over bandwidth
    t_math = batch * 2 * PARAMS / H100_FLOPS                                  # 2 FLOPs per parameter per token
    print(f"batch {batch:>2}: {t_mem * 1e3:4.1f} ms per step (memory), "
          f"{t_math * 1e3:5.3f} ms (maths), {batch / t_mem:,.0f} tokens/s (estimated)")
maths_t = 2 * PARAMS / H100_FLOPS
memory_t = (WEIGHT_READ + CONTEXT * KV_BYTES_PER_TOKEN) / H100_BW
print(f"maths share of a batch-1 step: {maths_t / memory_t:.1%}")

batch  1:  4.8 ms per step (memory), 0.016 ms (maths), 206 tokens/s (estimated)
batch 64:  9.3 ms per step (memory), 1.035 ms (maths), 6,894 tokens/s (estimated)
maths share of a batch-1 step: 0.3%


**What you should see:** at batch 1 the GPU's maths is well under 1% of each step, so the step is memory-bound.


How to read the numbers: 2 × parameters is the FLOPs needed per token (one multiply and one add per weight). Memory time is weights plus KV reads, divided by bandwidth. From batch 1 to batch 64, memory time only about doubles (4.8 to 9.3 ms), because the weights are read once for all 64 requests; only the KV reads grow with the batch. So tokens per second rise about 33x (206 to 6,894, estimated). This is why an engine batches. Each batched request still adds its own KV cache.

**Step 6: the same request through an engine (optional).** The client code is the same few lines you would write for any web API. The engine does the batching, the KV blocks and the prefix reuse on the server. This cell needs an engine running (lesson 08 starts one, tagged `needs-docker`). If none is reachable, it prints a notice and the estimates above still stand.

In [6]:
from openai import OpenAI

base = env("VLLM_BASE_URL", "http://localhost:8000/v1")   # lab value: the local engine from lesson 08
policy = "\n".join(f"Rule {i}: answer from the Brightside refund policy." for i in range(150))  # stand-in text
try:
    client = OpenAI(base_url=base, api_key="local")       # the local engine does not check the key
    model = client.models.list().data[0].id
    reply = client.chat.completions.create(
        model=model, max_tokens=60,
        messages=[{"role": "system", "content": policy},
                  {"role": "user", "content": "Can a customer get a refund after 30 days?"}])
    print(reply.choices[0].message.content[:200])
    print("prompt tokens:", reply.usage.prompt_tokens, "| answer tokens:", reply.usage.completion_tokens)
except Exception as err:   # the endpoint is optional here; print the error type, never the address
    print(f"No engine reachable ({type(err).__name__}): skipped. Start one in lesson 08.")

⏭  Skipped: this cell needs [docker]. Run it yourself, or re-run with TUT_ENABLE=docker.


**What you should see without an engine:** the runner reports the cell as skipped (`needs docker`). If it runs without the skip and no engine answers, the cell prints `No engine reachable (...): skipped. Start one in lesson 08.` No model answer is shown here, because none was recorded.

The 150 `Rule` lines are a stand-in policy of about 1,500 tokens, not the real Brightside policy. When an engine runs, the printed `prompt tokens` gives the exact count.

## At work
- **Cloudflare (May 2026)** split prefill and decode onto different machines. p90 inter-token latency fell from about 100 ms to 20–30 ms, about 3x. → lesson 34.
- **Meta with PyTorch and vLLM (Sep 2025)** reached 40–50% prefix-cache hits at 90% GPU memory (HBM) use with sticky routing. A cache helps only when the same prefix reaches the same server. → lessons 14 and 30.
- **Anthropic's postmortem (Sep 2025)** describes a routing change that sent short requests to servers built for a 1M-token window. At peak, 16% of requests in one hour were affected. Routing and config changes are quality risks. → lessons 30 and 35.
- **Databricks** defines memory-bandwidth utilisation as achieved over peak bandwidth; their example is 50% of a 2 TB/s peak. Measure your own deployment. → lesson 06.

**Sources:** Cloudflare, "High-performance LLMs": https://blog.cloudflare.com/high-performance-llms/ · Meta and PyTorch, "Disaggregated Inference at Scale with PyTorch & vLLM": https://pytorch.org/blog/disaggregated-inference-at-scale-with-pytorch-vllm/ · Anthropic, "A postmortem of three recent issues": https://www.anthropic.com/engineering/a-postmortem-of-three-recent-issues · Databricks, "LLM Inference Performance Engineering: Best Practices": https://www.databricks.com/blog/llm-inference-performance-engineering-best-practices

## Common mistakes
Mixing units is the most common slip: the cache is bytes per token, so you multiply by a token count. This cell adds a string to a byte count.

In [7]:
KV_BYTES_PER_TOKEN + "1600 tokens"

TypeError: unsupported operand type(s) for +: 'int' and 'str'

**The fix:** multiply by the count, `KV_BYTES_PER_TOKEN * 1_600`, then print the result with its unit.

## Try it
Change the answer to 800 tokens and compute the two token-pass counts for a 1,600-token prompt. Reuse `token_passes` from step 2. How does the ratio change, and why?

In [8]:
# your code here: reuse token_passes from step 2

**Answer** — try it first, then expand the hidden cell below.

In [9]:
for answer in (200, 800):
    no = token_passes(1_600, answer, kv_cache=False)
    yes = token_passes(1_600, answer, kv_cache=True)
    print(f"answer {answer}: {no:,} without, {yes:,} with, ratio {no / yes:,.0f}x")
# The ratio grows with answer length: without notes, the cost per new token grows with the text so far.

answer 200: 339,900 without, 1,800 with, ratio 189x
answer 800: 1,599,600 without, 2,400 with, ratio 666x


## Where the course goes
| Part | Level | Lessons | The question it answers |
|---|---|---|---|
| 1 · One server | L1 | 01–09 | What does one request cost, and how do I call a server? |
| 2 · Memory, batching and engines | L3 | 10–18 | How do I fit more requests into the same GPU? |
| 3 · Quantization, parallelism, hardware | L4 | 19–25 | How do I fit a 70B model and pick a GPU? |
| 4 · Capacity and economics | L5 | 26–28 | How many replicas, and what does a million tokens cost? |
| 5 · Serving at scale and operations | L5 | 29–37 | How do I route, scale, roll out and protect the service? |
| 6 · Enterprise and the capstone | L6 | 38–40 | How do I design a multi-region platform and defend it? |

The running example grows through the same six parts: one request, then a tuned engine, then a 70B model on several GPUs, then capacity for 10,000 QPS, then replicas and rollouts, then a multi-region design.

## Recap
- A request has two phases: **prefill** (the prompt in one parallel pass, which sets TTFT) and **decode** (one token per step, which sets TPOT).
- Decode reads all the weights every step, so it is **memory-bound** (limited by reads from GPU memory, not maths): at batch 1 the maths sits mostly idle.
- The **KV cache** keeps earlier tokens so they are not recomputed. It takes 128 KiB per token for FP16 Llama 3.1 8B and holds GPU memory until the request ends.
- Capacity is set by memory for tokens in flight, not requests per second. Storing the policy prefix once gives about 6x more requests (estimated).


**Interview:** "Why can't an LLM endpoint scale like a stateless web service?" Strong answer: each request's cost grows with its prompt and output and holds GPU memory (its KV cache) until it ends; decode is memory-bound, so you size by tokens in flight, batch with a scheduler, and watch TTFT and TPOT.